# 04 — Classification: Do Extracted Signals Improve Prediction?

Predicts `Company response to consumer` from credit reporting complaints, building up from a structured-fields-only baseline through topic labels (Notebook 02), entity features (Notebook 03), and finally narrative text itself via a fine-tuned transformer vs. a distilled version — an accuracy-vs-latency comparison.

**Question this notebook answers:** do any of the signals extracted in Notebooks 02 and 03 — or the raw narrative text itself — actually improve prediction beyond what the structured fields alone already capture? Each stage below only earns a place in the final comparison if it beats what came before it; a stage adding nothing is a real, reportable finding, not a failure to find something more exciting.


In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/ANLP_portfolio/")


Mounted at /content/drive/


In [30]:
import pandas as pd
import numpy as np
import warnings
import torch
import torch.nn as nn

warnings.filterwarnings('ignore', category=DeprecationWarning)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score


## Assembling the Working Dataset

Merging three files: the deduplicated credit reporting base (Notebook 01/02/03's working dataset), Notebook 02's topic assignments, and Notebook 03's final entity features. Using the deduplicated version (164,533 rows), consistent with Notebooks 02 and 03 — this also matters here for a different reason than it did for topic modeling: without deduplication, the same near-identical template narrative could land in both the train and test split, inflating apparent performance without reflecting real generalization.

In [7]:
BASE_PATH = "data/01_complaints_credit_reporting.csv"
TOPICS_PATH = "data/02_complaints_topics.csv"
ENTITIES_PATH = "data/03_complaints_entities_final.csv"

df = pd.read_csv(BASE_PATH, low_memory=False)
df = df.drop_duplicates(subset='Consumer complaint narrative').reset_index(drop=True)
print(f"Loaded from Notebook 01 ({BASE_PATH}): {df.shape}")

topics = pd.read_csv(TOPICS_PATH)
print(f"Loaded from Notebook 02 ({TOPICS_PATH}): {topics.shape}")

entities = pd.read_csv(ENTITIES_PATH)
print(f"Loaded from Notebook 03 ({ENTITIES_PATH}): {entities.shape}")

df = df.merge(topics, on='Complaint ID', how='left').merge(entities, on='Complaint ID', how='left')

print()
print(f"Merged working dataset: {df.shape}")
print(f"Rows missing a topic assignment (Notebook 02): {df['dominant_topic'].isna().sum()}")
print(f"Rows missing entity features (Notebook 03): {df['has_dollar_amount'].isna().sum()}")

Loaded from Notebook 01 (data/01_complaints_credit_reporting.csv): (164533, 16)
Loaded from Notebook 02 (data/02_complaints_topics.csv): (164533, 3)
Loaded from Notebook 03 (data/03_complaints_entities_final.csv): (164533, 6)

Merged working dataset: (164533, 23)
Rows missing a topic assignment (Notebook 02): 0
Rows missing entity features (Notebook 03): 0


## Consolidating the Issue Field

Notebook 02 found two pairs of `Issue` values that are renamed duplicates of the same category (differing only in wording, not real category). Reapplying that same consolidation here, since `Issue` is a candidate structured feature below and should be as clean as it was for the topic-vs-issue comparison.

In [8]:
ISSUE_MERGE_MAP = {
    "Problem with a company's investigation into an existing issue": "Problem with a company's investigation into an existing problem",
    "Identity theft protection or other monitoring services": "Credit monitoring or identity theft protection services",
}
df['Issue'] = df['Issue'].replace(ISSUE_MERGE_MAP)
print(df['Issue'].value_counts())


Issue
Incorrect information on your report                               102524
Improper use of your report                                         34980
Problem with a company's investigation into an existing problem     24333
Credit monitoring or identity theft protection services               976
Unable to get your credit report or credit score                      948
Problem with fraud alerts or security freezes                         772
Name: count, dtype: int64


## Defining the Target

Confirmed in Notebook 04's planning: `Consumer disputed?` doesn't exist in this dataset's column list — CFPB discontinued it before this project's July 2025–June 2026 window. `Company response to consumer` is the target, no longer an open question.

In [9]:
TARGET_COL = 'Company response to consumer'

print(df[TARGET_COL].value_counts(normalize=True))


Company response to consumer
Closed with explanation            0.638144
Closed with non-monetary relief    0.358256
Untimely response                  0.002353
Closed with monetary relief        0.001155
In progress                        0.000091
Name: proportion, dtype: float64


### Target Distribution

Scoped to credit reporting: Closed with explanation (63.8%), Closed with non-monetary relief (35.8%), Untimely response (0.24%), Closed with monetary relief (0.12%), In progress (0.01%). Monetary relief is much rarer here than in the broader complaint population (1.3–2.2% across all products) — makes sense, credit-reporting disputes usually get resolved by fixing the record, not paying anyone.

## Baseline 1: Structured Fields Only

`Product` isn't included — this dataset is scoped to credit reporting only, so it's constant and carries no information. Using `Issue`, `Sub-issue`, `State`, `Submitted via`, and `Timely response?` instead — the fields that actually vary within this scope.

In [10]:
print(f"Missing values in target: {df[TARGET_COL].isna().sum()} of {len(df)}")

Missing values in target: 78 of 164533


78 of 164,533 rows (0.05%) have no recorded `Company response to consumer` — likely complaints still in process at the time of this data pull, not a data quality problem. Dropping them; the loss is negligible.

In [11]:
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
print(f"Shape after dropping missing targets: {df.shape}")

Shape after dropping missing targets: (164455, 23)


In [12]:
# Save the udpated df
df.to_csv("data/04_complaints_model_ready.csv", index=False)
print(f"Saved from Notebook 04: {df.shape}")

Saved from Notebook 04: (164455, 23)


In [13]:
# df is updated version here.
# uncomment below to reload from local if don't want to start from begining

df = pd.read_csv("data/04_complaints_model_ready.csv", low_memory=False)
print(f"Loaded from Notebook 04: {df.shape}")

Loaded from Notebook 04: (164455, 23)


In [14]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

structured_cols = ['Issue', 'Sub-issue', 'State', 'Submitted via', 'Timely response?']
X = df[structured_cols].fillna('Unknown')
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

ct = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), structured_cols)])
baseline = Pipeline([
    ('prep', ct),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)
print(classification_report(y_test, baseline_preds))
print(f"Macro F1: {f1_score(y_test, baseline_preds, average='macro'):.3f}")


                                 precision    recall  f1-score   support

        Closed with explanation       0.66      0.28      0.39     20989
    Closed with monetary relief       0.00      0.66      0.01        38
Closed with non-monetary relief       0.38      0.47      0.42     11784
                    In progress       0.00      0.33      0.00         3
              Untimely response       0.61      1.00      0.76        77

                       accuracy                           0.35     32891
                      macro avg       0.33      0.55      0.32     32891
                   weighted avg       0.56      0.35      0.40     32891

Macro F1: 0.315


In [15]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_preds = dummy.predict(X_test)

print(classification_report(y_test, dummy_preds, zero_division=0))
print(f"Macro F1: {f1_score(y_test, dummy_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.64      1.00      0.78     20989
    Closed with monetary relief       0.00      0.00      0.00        38
Closed with non-monetary relief       0.00      0.00      0.00     11784
                    In progress       0.00      0.00      0.00         3
              Untimely response       0.00      0.00      0.00        77

                       accuracy                           0.64     32891
                      macro avg       0.13      0.20      0.16     32891
                   weighted avg       0.41      0.64      0.50     32891

Macro F1: 0.156


## Model Evaluation: Class Imbalance Diagnosis

I ran the classifier and it collapsed to predicting only the majority class, "Closed with explanation." Recall on that class is 1.00, and recall on every other class is 0.00, so the 0.64 accuracy is just the base rate of the majority class in the dataset (20,989 / 32,891 = 0.638).

The class distribution is heavily skewed:

| Class | Support |
|---|---|
| Closed with explanation | 20,989 |
| Closed with non-monetary relief | 11,784 |
| Untimely response | 77 |
| Closed with monetary relief | 38 |
| In progress | 3 |

Macro F1 came out to 0.156, against a weighted F1 of 0.50. The gap between these two numbers is the diagnostic: weighted F1 is propped up by the majority class alone, while macro F1 treats all five classes equally and exposes that the model has learned nothing beyond guessing the most frequent label.

**Next step:** I'm addressing this with class weighting (`class_weight='balanced'`) rather than synthetic oversampling. SMOTE generates synthetic samples by interpolating between a point and its nearest neighbors of the same class, and with only 3 samples in "In progress" and 38 in "Closed with monetary relief," there isn't enough real signal for that interpolation to be meaningful. Class weighting reweights the loss by inverse class frequency instead, so training stays grounded in real samples. I'm also merging "In progress" (n=3) and "Untimely response" (n=77) into a single "Other" category — at that sample size, modeling them as separate decision boundaries isn't defensible.

## Refitting on Merged Classes

Cell 18 diagnosed the collapse to majority-class prediction. Before trying different weighting schemes, the more basic problem underneath it needs addressing first: two classes ("In progress" at n=3, "Untimely response" at n=77) don't have enough support to define a decision boundary at all, regardless of how the loss is weighted. Merging them into a single "Other" category, then refitting with `class_weight='balanced'` to see what weighting alone can do once the unsalvageable classes are out of the way.


In [16]:
# Merging sparse response categories before refitting.
# "In progress" (n=3) and "Untimely response" (n=77) don't have enough
# support to learn a separate decision boundary — class_weight='balanced'
# can upweight them, but it can't manufacture signal that isn't there.
TARGET_MERGE_MAP = {
    'In progress': 'Other',
    'Untimely response': 'Other',
}
y_merged = y.replace(TARGET_MERGE_MAP)
print(y_merged.value_counts())

Company response to consumer
Closed with explanation            104946
Closed with non-monetary relief     58917
Other                                 402
Closed with monetary relief           190
Name: count, dtype: int64


In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)

ct = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), structured_cols)])
baseline_merged = Pipeline([
    ('prep', ct),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
baseline_merged.fit(X_train, y_train)

baseline_merged_preds = baseline_merged.predict(X_test)
print(classification_report(y_test, baseline_merged_preds))
print(f"Macro F1: {f1_score(y_test, baseline_merged_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.66      0.28      0.39     20989
    Closed with monetary relief       0.00      0.71      0.01        38
Closed with non-monetary relief       0.38      0.53      0.44     11784
                          Other       0.34      0.99      0.51        80

                       accuracy                           0.37     32891
                      macro avg       0.35      0.63      0.34     32891
                   weighted avg       0.56      0.37      0.41     32891

Macro F1: 0.339


## Refit with Merged Classes: Improvement and a New Problem

Merging "In progress" and "Untimely response" into "Other" and refitting with `class_weight='balanced'` pushed macro F1 from 0.156 to 0.338. The model now picks up real signal on "Closed with non-monetary relief" — recall went from 0.00 to 0.53, with 0.38 precision.

This came at a cost. Accuracy dropped from 0.64 to 0.37 and recall on the majority class fell from 1.00 to 0.28. `class_weight='balanced'` shifts the decision boundary toward minority classes, and pulling performance away from the majority class is the mechanical price of that shift, not a defect in the fit.

"Closed with monetary relief" is the class to flag. Recall is 0.71 but precision rounds to 0.00 — F1 lands at 0.01. With only 38 training samples, inverse-frequency weighting assigns this class a weight around 550x the majority class (20989/38). That's enough to make the model over-predict "Closed with monetary relief" into other classes' territory just to catch the few true positives, which tanks precision everywhere it fires.

Balanced weighting fixed the collapse but overcorrected on the smallest class. Next step is capping the weight manually instead of using `'balanced'` — something closer to `sqrt`-scaled inverse frequency — so the minority classes get boosted without swamping the decision boundary.

In [18]:
from collections import Counter

# Sqrt-scaled inverse frequency instead of 'balanced' (which is linear
# inverse frequency). Linear scaling gave "Closed with monetary relief"
# (n≈152 in train) a weight ~550x the majority class, which pushed the
# model to over-predict it — recall 0.71, precision ~0.00. Sqrt-scaling
# still upweights the minority classes but caps how aggressive that gets.
counts = Counter(y_train)
total = sum(counts.values())
n_classes = len(counts)

sqrt_weights = {
    cls: np.sqrt(total / (n_classes * count))
    for cls, count in counts.items()
}
print(sqrt_weights)

{'Closed with explanation': np.float64(0.6259073954297711), 'Closed with non-monetary relief': np.float64(0.8353644650625962), 'Other': np.float64(10.106728587080948), 'Closed with monetary relief': np.float64(14.710137929154058)}


In [19]:
ct = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), structured_cols)])
baseline_sqrt = Pipeline([
    ('prep', ct),
    ('clf', LogisticRegression(max_iter=1000, class_weight=sqrt_weights)),
])
baseline_sqrt.fit(X_train, y_train)

sqrt_preds = baseline_sqrt.predict(X_test)
print(classification_report(y_test, sqrt_preds))
print(f"Macro F1: {f1_score(y_test, sqrt_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.64      0.97      0.77     20989
    Closed with monetary relief       0.03      0.05      0.03        38
Closed with non-monetary relief       0.46      0.03      0.06     11784
                          Other       0.62      0.99      0.76        80

                       accuracy                           0.64     32891
                      macro avg       0.44      0.51      0.41     32891
                   weighted avg       0.57      0.64      0.52     32891

Macro F1: 0.406


## Sqrt-Scaled Weighting: Higher Macro F1, but Not a Real Fix

Sqrt-scaled inverse frequency weights (`{Closed with explanation: 0.63, Closed with non-monetary relief: 0.84, Other: 10.1, Closed with monetary relief: 14.7}`) pushed macro F1 to 0.406, up from 0.338 under `'balanced'`. The number looks better, but the per-class breakdown tells a different story.

"Closed with non-monetary relief" recall collapsed from 0.53 (under `'balanced'`) to 0.03. That's my second-largest class at 11,784 samples, and the model now almost never predicts it. The sqrt weights for "Closed with explanation" (0.63) and "Closed with non-monetary relief" (0.84) are close enough that the model reverted to favoring the class it already leans toward from sheer representation — accuracy climbing back to 0.64 backs this up, landing suspiciously close to the original collapsed baseline.

"Closed with monetary relief" didn't get meaningfully fixed either. Precision moved from ~0.00 to 0.03 and recall dropped from 0.71 to 0.05 — F1 basically held flat at 0.01 to 0.03. Sqrt-scaling capped the over-prediction problem from the `'balanced'` run, but capped it so hard the class is functionally being ignored again.

"Other" stayed strong, recall at 0.99, consistent with the prior run.

The macro F1 gain here is coming from a different failure mode, not a genuine improvement — the model traded away non-monetary-relief recall for a milder monetary-relief precision problem, and averaging across classes hides that trade. Macro F1 alone doesn't tell me which failure mode is actually worse for this task. Next step is trying a weighting scheme between linear and sqrt — an intermediate exponent or a hand-set cap — aiming to keep the non-monetary-relief recall from the `'balanced'` run while reining in the monetary-relief precision problem, instead of just swapping one collapse for another.

##Hand-Capped Weighting

Sqrt-scaling raised macro F1, but for the wrong reason — it compressed the weights for the two largest classes close enough together that the model reverted to favoring the class it already leans toward from raw representation, collapsing "Closed with non-monetary relief" recall in the process. Trying a middle ground: keep the linear inverse-frequency weighting from `'balanced'`, but cap the maximum multiplier so no single class (particularly "Closed with monetary relief," weighted ~550x under `'balanced'`) can dominate the decision boundary.

In [20]:
# Hand-capped weighting: linear inverse-frequency, but capped at a max
# multiplier so no single class can dominate the decision boundary the
# way "Closed with monetary relief" did under 'balanced' (~550x).
# Starting cap at 60x — high enough to meaningfully upweight the class,
# low enough that it shouldn't trigger the over-prediction seen before.
MAX_WEIGHT = 60

counts = Counter(y_train)
total = sum(counts.values())
n_classes = len(counts)

capped_weights = {
    cls: min(total / (n_classes * count), MAX_WEIGHT)
    for cls, count in counts.items()
}
print(capped_weights)

{'Closed with explanation': 0.3917600676536799, 'Closed with non-monetary relief': 0.6978337894893175, 'Other': 60, 'Closed with monetary relief': 60}


In [21]:
ct = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), structured_cols)])
baseline_capped = Pipeline([
    ('prep', ct),
    ('clf', LogisticRegression(max_iter=1000, class_weight=capped_weights)),
])
baseline_capped.fit(X_train, y_train)

capped_preds = baseline_capped.predict(X_test)
print(classification_report(y_test, capped_preds))
print(f"Macro F1: {f1_score(y_test, capped_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.67      0.34      0.45     20989
    Closed with monetary relief       0.01      0.34      0.02        38
Closed with non-monetary relief       0.38      0.68      0.49     11784
                          Other       0.33      0.99      0.50        80

                       accuracy                           0.46     32891
                      macro avg       0.35      0.59      0.37     32891
                   weighted avg       0.57      0.46      0.46     32891

Macro F1: 0.365


## Side-by-Side: Balanced vs. Sqrt-Scaled vs. Hand-Capped Weighting

| Metric | `'balanced'` (linear) | sqrt-scaled | hand-capped (max 60x) |
|---|---|---|---|
| Macro F1 | 0.338 | 0.406 | 0.361 |
| Accuracy | 0.37 | 0.64 | 0.46 |
| **Closed with explanation** (n=20,989) | P 0.66 / R 0.28 / F1 0.39 | P 0.64 / R 0.97 / F1 0.77 | P 0.67 / R 0.34 / F1 0.45 |
| **Closed with non-monetary relief** (n=11,784) | P 0.38 / R 0.53 / F1 0.44 | P 0.45 / R 0.03 / F1 0.06 | P 0.38 / R 0.68 / F1 0.49 |
| **Closed with monetary relief** (n=38) | P 0.00 / R 0.71 / F1 0.01 | P 0.03 / R 0.05 / F1 0.03 | P 0.01 / R 0.34 / F1 0.02 |
| **Other** (n=80) | P 0.34 / R 0.99 / F1 0.51 | P 0.62 / R 0.99 / F1 0.76 | P 0.32 / R 0.99 / F1 0.48 |

Highest macro F1 doesn't mean best model here. Sqrt-scaling wins on the headline number but for the wrong reason — its weights for the two big classes (0.63 and 0.84) are close enough that the model reverts to majority-class behavior, tanking non-monetary-relief recall to 0.03. That's the same collapse pattern from the original unweighted model, just partially masked by the "Other" and monetary-relief numbers padding the average.

Capped weighting is the one that actually behaves like a real fix rather than a different failure mode. It's the only version where **both** major classes hold reasonable recall at the same time — explanation at 0.34, non-monetary relief at 0.68 (its best score across all three runs) — instead of one collapsing to prop up the other. It also brings the monetary-relief false-positive problem down from `'balanced'`'s extreme (recall 0.71 on 550x weight, precision ~0.00) without fully collapsing it the way sqrt did.

The unresolved piece: the 60x cap is binding on both "Other" (n=80) and "Closed with monetary relief" (n=38) — two classes with very different frequencies collapsing to the same weight, which is worth a mention in the writeup since it means the cap is currently doing more work than the actual class ratio for those two.

Capped weighting is the most defensible middle ground of the three, even though it doesn't have the highest macro F1. I'd lock this in as the weighting scheme going into Baseline 2 rather than treat sqrt's higher number as the win.

In [22]:
# Persisting Baseline 1's full result — classification report included —
# for the five-stage comparison in the final summary table (cell 26),
# and saving `capped_weights` so the same weighting scheme carries into
# Baselines 2 and 3 without re-deriving it.
results_baseline1 = {
    'model': 'structured_only_capped_weights',
    'macro_f1': f1_score(y_test, capped_preds, average='macro'),
    'accuracy': (capped_preds == y_test).mean(),
    'class_weights': capped_weights,
    'classification_report': classification_report(y_test, capped_preds, output_dict=True),
}
print(results_baseline1)

import json
with open('data/04_baseline1_capped_results.json', 'w') as f:
    json.dump(results_baseline1, f, indent=2)

print(f"Saved to data/04_baseline1_capped_results.json")

{'model': 'structured_only_capped_weights', 'macro_f1': 0.36506626029459277, 'accuracy': np.float64(0.4624061293362926), 'class_weights': {'Closed with explanation': 0.3917600676536799, 'Closed with non-monetary relief': 0.6978337894893175, 'Other': 60, 'Closed with monetary relief': 60}, 'classification_report': {'Closed with explanation': {'precision': 0.674516159786443, 'recall': 0.33708132831483156, 'f1-score': 0.44952029989198805, 'support': 20989.0}, 'Closed with monetary relief': {'precision': 0.01145374449339207, 'recall': 0.34210526315789475, 'f1-score': 0.02216538789428815, 'support': 38.0}, 'Closed with non-monetary relief': {'precision': 0.38240608654303376, 'recall': 0.6824507807196198, 'f1-score': 0.49015664045834095, 'support': 11784.0}, 'Other': {'precision': 0.3333333333333333, 'recall': 0.9875, 'f1-score': 0.49842271293375395, 'support': 80.0}, 'accuracy': 0.4624061293362926, 'macro avg': {'precision': 0.3504273310390505, 'recall': 0.5872843430480865, 'f1-score': 0.36

In [ ]:
# To reload later if necessary, uncomment the code below:

# with open('data/04_baseline1_capped_results.json') as f:
#     results_baseline1 = json.load(f)

### Note

The intro below (originally written before the class-weighting decisions in cells 18–29) is superseded by the capped-weighting version that follows. Kept here only as a record of the original framing question; cell 33 is the version that actually applies to the code and result below.

## Baseline 2: Structured Fields + Topic Labels (Capped Weighting)

Running Baseline 2 with the weighting scheme locked in from Baseline 1 — `capped_weights` (max 60x inverse frequency) instead of `'balanced'`, and `y_merged` as the target instead of the original five-class target. Reusing the exact weight values from Baseline 1 rather than recomputing them from `y2_train`'s counts, since both splits come from the same `y_merged` with the same `random_state=42` and `stratify=y_merged` — the point of locking in a weighting scheme is comparability across stages, and recomputing per-stage would reintroduce the moving-target problem the capped weighting was meant to fix in the first place.

This tests whether `topic_name` (Notebook 02's k=7 LDA topics) adds anything beyond `Issue`/`Sub-issue` — but now against a fair baseline. Comparing this against Baseline 1's capped-weighting result (macro F1 0.361) rather than the original unweighted or `'balanced'` results, so the topic-label comparison isn't confounded by a different weighting scheme underneath it.

In [21]:
structured_plus_topic_cols = structured_cols + ['topic_name']
X2 = df[structured_plus_topic_cols].fillna('Unknown')

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)

ct2 = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), structured_plus_topic_cols)])
model_topic = Pipeline([
    ('prep', ct2),
    ('clf', LogisticRegression(max_iter=1000, class_weight= capped_weights)),
])
model_topic.fit(X2_train, y2_train)

topic_preds = model_topic.predict(X2_test)
print(classification_report(y2_test, topic_preds))
print(f"Macro F1: {f1_score(y2_test, topic_preds, average='macro'):.3f}")


                                 precision    recall  f1-score   support

        Closed with explanation       0.69      0.37      0.48     20989
    Closed with monetary relief       0.01      0.61      0.02        38
Closed with non-monetary relief       0.40      0.65      0.49     11784
                          Other       0.46      0.99      0.63        80

                       accuracy                           0.47     32891
                      macro avg       0.39      0.65      0.41     32891
                   weighted avg       0.58      0.47      0.49     32891

Macro F1: 0.406


## Baseline 2 Result: Topic Labels Add Real Signal

Adding `topic_name` (Notebook 02's k=7 LDA topics) to the capped-weighting baseline pushed macro F1 from 0.361 to 0.406. Unlike the sqrt-scaled weighting experiment, this gain isn't one class propping up the average while another collapses — most classes held or improved together, which is what a feature actually adding information looks like rather than a reshuffled decision boundary.

Per-class, against Baseline 1's capped result:
- Closed with explanation: F1 0.45 → 0.48
- Closed with non-monetary relief: F1 0.49 → 0.49, essentially flat
- Other: F1 0.48 → 0.63, the clearest gain — precision moved 0.32 → 0.46
- Closed with monetary relief: F1 0.02 → 0.02, recall roughly doubled (0.34 → 0.61) but precision stayed near 0.01

"Other" is where `topic_name` earns its keep most — the topic labels separate that small group more cleanly than `Issue`/`Sub-issue` alone. The monetary-relief problem is unchanged in kind: near-zero precision regardless of recall. At n=38 that's a sample-size ceiling, not something any feature set fixes on its own.

Answering the question this baseline was built to test: `topic_name` adds real, if modest, signal beyond `Issue`/`Sub-issue` — not the "largely redundant" outcome Notebook 02's topic-vs-issue cross-tab made plausible going in.

In [22]:
# Persisting Baseline 2's full result for the five-stage comparison in
# the final summary table (cell 26).
results_baseline2 = {
    'model': 'structured_plus_topic_capped_weights',
    'macro_f1': f1_score(y2_test, topic_preds, average='macro'),
    'accuracy': (topic_preds == y2_test).mean(),
    'class_weights': capped_weights,
    'classification_report': classification_report(y2_test, topic_preds, output_dict=True),
}
print(results_baseline2)

import json
with open('data/04_baseline2_capped_results.json', 'w') as f:
    json.dump(results_baseline2, f, indent=2)

print("Saved to data/04_baseline2_capped_results.json")

{'model': 'structured_plus_topic_capped_weights', 'macro_f1': 0.4056633717494055, 'accuracy': np.float64(0.47213523456264633), 'class_weights': {'Closed with explanation': 0.3917600676536799, 'Closed with non-monetary relief': 0.6978337894893175, 'Other': 60, 'Closed with monetary relief': 60}, 'classification_report': {'Closed with explanation': {'precision': 0.6854269583627381, 'recall': 0.3701939110962885, 'f1-score': 0.48074245939675175, 'support': 20989.0}, 'Closed with monetary relief': {'precision': 0.010564997703261369, 'recall': 0.6052631578947368, 'f1-score': 0.02076749435665914, 'support': 38.0}, 'Closed with non-monetary relief': {'precision': 0.39867749661564095, 'recall': 0.6497793618465716, 'f1-score': 0.4941594062600839, 'support': 11784.0}, 'Other': {'precision': 0.45930232558139533, 'recall': 0.9875, 'f1-score': 0.626984126984127, 'support': 80.0}, 'accuracy': 0.47213523456264633, 'macro avg': {'precision': 0.38849294456575895, 'recall': 0.6531841077093992, 'f1-score'

## Baseline 3: Structured Fields + Entity Features

Adding Notebook 03's `has_dollar_amount`, `dollar_amount_count`, `has_company_org`, `company_org_count`, `has_date`. These went through four failed filtering attempts before landing on a row-level self-match against each complaint's own `Company` field — worth seeing whether that effort actually pays off in prediction, not just in a cleaner-looking chart.

In [23]:
entity_cols = ['has_dollar_amount', 'dollar_amount_count', 'has_company_org', 'company_org_count', 'has_date']
structured_plus_entity_cols = structured_cols + entity_cols

X3 = df[structured_plus_entity_cols].copy()
for col in ['Issue', 'Sub-issue', 'State', 'Submitted via', 'Timely response?']:
    X3[col] = X3[col].fillna('Unknown')

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)

ct3 = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), structured_cols),
    ('num', 'passthrough', entity_cols),
])
model_entities = Pipeline([
    ('prep', ct3),
    ('clf', LogisticRegression(max_iter=1000, class_weight=capped_weights)),
])
model_entities.fit(X3_train, y3_train)

entity_preds = model_entities.predict(X3_test)
print(classification_report(y3_test, entity_preds))
print(f"Macro F1: {f1_score(y3_test, entity_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.68      0.54      0.60     20989
    Closed with monetary relief       0.01      0.42      0.03        38
Closed with non-monetary relief       0.41      0.51      0.45     11784
                          Other       0.33      0.99      0.49        80

                       accuracy                           0.53     32891
                      macro avg       0.36      0.62      0.39     32891
                   weighted avg       0.58      0.53      0.55     32891

Macro F1: 0.394


## Baseline 3 Result: Entity Features Improve Overall Accuracy, But Add Limited Minority-Class Signal

Testing Notebook 03's entity features (`has_dollar_amount`, `dollar_amount_count`, `has_company_org`, `company_org_count`, `has_date`) against the same capped-weighting scheme used in Baselines 1 and 2. Macro F1 landed at 0.394 — below Baseline 2's 0.406, but the comparison isn't apples-to-apples on what's driving it.

Against Baseline 1 (capped, no extra features), entity features produced the largest single-class gain seen so far: "Closed with explanation" F1 went from 0.45 to 0.60, recall alone moving 0.34 → 0.54. Accuracy climbed from 0.46 to 0.53, also the biggest jump of any baseline. "Closed with non-monetary relief" dipped slightly (F1 0.49 → 0.45) and "Other" stayed flat. "Closed with monetary relief" is still functionally broken — precision near 0.01 — but recall at 0.42 with that precision floor is the least-bad version of this class across all three baselines.

Against Baseline 2 (capped + `topic_name`), the two feature sets are pulling weight in opposite places. Entity features boost the majority class (F1 0.60 vs 0.48 with topic labels); topic labels boost "Other" (F1 0.63 vs 0.49 with entity features). Macro F1 penalizes Baseline 3 slightly because it weights "Other" (n=80) equally with "Closed with explanation" (n=20,989), despite the massive support gap.

Neither feature set is a strict win — entity features give a stronger majority-class model with better overall accuracy, topic labels give a stronger small-class model with better macro F1. That's the natural setup for testing whether stacking both together beats either alone.

## Baseline 3b: Combined Topic + Entity Features (Supplementary)

Baseline 2 (topic labels) and Baseline 3 (entity features) helped in different, non-overlapping ways — topic labels lifted the "Other" class, entity features lifted the majority class. Testing whether stacking both together compounds those gains, or whether the two feature sets are drawing on the same underlying signal instead of complementary signal. This isn't one of the notebook's five official stages (see the summary table at the end), so it's kept as a supplementary check rather than folded into the main numbering.

In [24]:
structured_plus_topic_plus_entity_cols = structured_cols + ['topic_name'] + entity_cols

X4 = df[structured_plus_topic_plus_entity_cols].copy()
for col in structured_cols + ['topic_name']:
    X4[col] = X4[col].fillna('Unknown')

X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4, y_merged, test_size=0.2, random_state=42, stratify=y_merged
)

ct4 = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), structured_cols + ['topic_name']),
    ('num', 'passthrough', entity_cols),
])
model_combined = Pipeline([
    ('prep', ct4),
    ('clf', LogisticRegression(max_iter=1000, class_weight=capped_weights)),
])
model_combined.fit(X4_train, y4_train)

combined_preds = model_combined.predict(X4_test)
print(classification_report(y4_test, combined_preds))
print(f"Macro F1: {f1_score(y4_test, combined_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.70      0.47      0.56     20989
    Closed with monetary relief       0.01      0.53      0.02        38
Closed with non-monetary relief       0.42      0.60      0.49     11784
                          Other       0.33      0.97      0.50        80

                       accuracy                           0.52     32891
                      macro avg       0.37      0.64      0.39     32891
                   weighted avg       0.60      0.52      0.54     32891

Macro F1: 0.393


## Baseline 3b Result: Combined Topic + Entity Features Don't Beat Either Alone

Stacking `topic_name` and the entity features together, I got macro F1 0.393 — essentially tied with entity features alone (0.394) and below topic labels alone (0.406). Combining the two feature sets didn't give me a synergy; it gave me an average.

| | Baseline 1 (structured only) | Baseline 2 (+topic) | Baseline 3 (+entity) | Baseline 3b (+topic+entity) |
|---|---|---|---|---|
| Macro F1 | 0.361 | 0.406 | 0.394 | 0.393 |
| Accuracy | 0.46 | 0.47 | 0.53 | 0.52 |
| Explanation F1 | 0.45 | 0.48 | 0.60 | 0.56 |
| Non-monetary relief F1 | 0.49 | 0.49 | 0.45 | 0.49 |
| Other F1 | 0.48 | 0.63 | 0.49 | 0.50 |
| Monetary relief F1 | 0.02 | 0.02 | 0.03 | 0.02 |

Every class in the combined model lands between its two parent scores rather than matching or beating the best of the two. Explanation F1 (0.56) falls between topic's 0.48 and entity's 0.60. Non-monetary relief F1 (0.49) matches topic's number, not entity's lower one. "Other" F1 (0.50) sits close to entity's 0.49, well under topic's standout 0.63. I read this as the two feature sets capturing overlapping signal rather than complementary signal — where they'd push the model in different directions, it settles for a blend instead of the best of both.

I'm treating this as a legitimate finding, not a failed experiment: structured fields plus one well-chosen feature set already captures most of what's available in this feature space, and stacking more on top shows diminishing returns rather than continued improvement. Which single feature set wins depends on what I'm optimizing for — topic labels for macro F1 and small-class performance, entity features for majority-class accuracy and overall accuracy. This sets up the open question I need Baseline 5 (transformer) to answer: does the raw narrative text carry information beyond anything captured by `Issue`, `Sub-issue`, topic assignments, or entity flags — or does the ceiling on this task sit lower than narrative modeling can push past?

## Baseline 4: Transformer Fine-Tune on Narrative Text

In [23]:
# Recovering the exact same train/test row split used by every baseline —
# train_test_split with the same length input, same y_merged, and same
# random_state=42 always produces the same indices, regardless of what X is.
all_idx = np.arange(len(df))
train_idx, test_idx = train_test_split(
    all_idx, test_size=0.2, random_state=42, stratify=y_merged
)
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}")

# Compute constraint from cell 22's own TODO note: subsample the TRAINING
# set to ~25,000 rows, stratified, so fine-tuning is feasible. Evaluation
# stays on the FULL test set (32,891 rows) so macro F1 is directly
# comparable to Baselines 1–4, none of which were subsampled.
TRAIN_SUBSAMPLE_SIZE = 25000
y_merged_train = y_merged.iloc[train_idx]

train_sub_idx, _ = train_test_split(
    train_idx, train_size=TRAIN_SUBSAMPLE_SIZE,
    random_state=42, stratify=y_merged_train
)
print(f"Subsampled train size: {len(train_sub_idx)}")
print(y_merged.iloc[train_sub_idx].value_counts())

Train: 131564, Test: 32891
Subsampled train size: 25000
Company response to consumer
Closed with explanation            15954
Closed with non-monetary relief     8956
Other                                 61
Closed with monetary relief           29
Name: count, dtype: int64


In [25]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [26]:
from datasets import Dataset
import torch.nn as nn

label_list = sorted(y_merged.unique())
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

train_texts = df.loc[train_sub_idx, 'Consumer complaint narrative'].fillna('').tolist()
train_labels = [label2id[l] for l in y_merged.loc[train_sub_idx]]
test_texts = df.loc[test_idx, 'Consumer complaint narrative'].fillna('').tolist()
test_labels = [label2id[l] for l in y_merged.loc[test_idx]]

train_ds = Dataset.from_dict({'text': train_texts, 'label': train_labels})
test_ds = Dataset.from_dict({'text': test_texts, 'label': test_labels})

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_ds = train_ds.map(tokenize_fn, batched=True).rename_column('label', 'labels')
test_ds = test_ds.map(tokenize_fn, batched=True).rename_column('label', 'labels')
train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/32891 [00:00<?, ? examples/s]

In [31]:
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

In [32]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id
)

# Carrying capped_weights into the loss so the weighting scheme stays
# identical to Baselines 1–4 instead of quietly training unweighted.
class_weight_tensor = torch.tensor(
    [capped_weights[id2label[i]] for i in range(len(label_list))], dtype=torch.float
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=class_weight_tensor.to(outputs.logits.device))
        loss = loss_fct(outputs.logits.view(-1, len(label_list)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    report = classification_report(labels, preds, target_names=label_list, output_dict=True, zero_division=0)
    return {'macro_f1': report['macro avg']['f1-score'], 'accuracy': report['accuracy']}

training_args = TrainingArguments(
    output_dir='./distilbert_transformer_stage',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.577715,0.929594,0.386934,0.596577
2,0.628286,0.870973,0.402085,0.653796
3,0.640191,0.965754,0.409159,0.660758


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4689, training_loss=0.729483691683505, metrics={'train_runtime': 88.8357, 'train_samples_per_second': 844.256, 'train_steps_per_second': 52.783, 'total_flos': 4967704627200000.0, 'train_loss': 0.729483691683505, 'epoch': 3.0})

In [33]:
preds_output = trainer.predict(test_ds)
transformer_preds = [id2label[i] for i in np.argmax(preds_output.predictions, axis=1)]

print(classification_report(y_merged.loc[test_idx], transformer_preds))
print(f"Macro F1: {f1_score(y_merged.loc[test_idx], transformer_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.80      0.63      0.70     20989
    Closed with monetary relief       0.00      0.00      0.00        38
Closed with non-monetary relief       0.52      0.72      0.61     11784
                          Other       0.36      0.30      0.33        80

                       accuracy                           0.66     32891
                      macro avg       0.42      0.41      0.41     32891
                   weighted avg       0.70      0.66      0.67     32891

Macro F1: 0.409


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Baseline 4: Transformer Fine-Tune (DistilBERT) on Narrative Text

Fine-tuned `distilbert-base-uncased` on the consumer complaint narratives for 3 epochs, using the same `capped_weights` and `y_merged` target as Baselines 1–3. Macro F1 came in at 0.409 — essentially tied with Baseline 2's 0.406, not the clear jump I might have expected from moving to raw text.

**This isn't a clean comparison to Baselines 1–3.** Those trained on the full ~131,564-row training set; the transformer trained on a 25,000-row stratified subsample, per the compute-cost tradeoff flagged in cell 22. Stratified sampling holds proportions steady but collapses absolute counts for the smallest classes — "Closed with monetary relief" dropped from roughly 152 examples in the full training set to 29, "Other" from roughly 320 to 61. Whatever this stage shows about narrative text, it's confounded by training on far less data for the classes that already had the least to work with.

**A result I need to check before drawing conclusions from it:** recall on "Closed with monetary relief" is 0.00. Every earlier model, even the overcorrected `'balanced'` run, predicted this class at least some of the time — this one never does, despite the same 60x loss weight being in place. I don't yet know if 29 training examples is simply below the floor for this architecture to carve out a decision region, or if the model is defaulting to never predicting a class it saw too rarely to trust. Next step is pulling the raw logits on a few of these examples before I treat this as a finding about narrative text rather than an artifact of the subsample.

**What genuinely improved:** "Closed with non-monetary relief" hit F1 0.61 — the best result of any baseline so far, with precision (0.52) and recall (0.72) both up together rather than trading off. "Closed with explanation" precision reached 0.80, also a high across all baselines, though recall fell to 0.63 to get there. "Other" dropped to F1 0.33, down from 0.48–0.63 in the structured-feature baselines — consistent with the subsampling problem, since 61 training examples is thin even for a model that reads the text directly.

**Training curve note:** training loss ticked up slightly each epoch (0.578 → 0.628 → 0.640) and validation loss moved non-monotonically (0.930 → 0.871 → 0.966), while macro F1 kept improving anyway. Not a red flag on its own — `load_best_model_at_end=True` selected the best checkpoint — but not a textbook-clean curve either, worth a one-line mention rather than treating training as unremarkable.

**Bottom line:** narrative text gets a real, specific win on "Closed with non-monetary relief" that no structured-feature baseline matched. It doesn't get an unambiguous overall win, and the comparison is muddied by the training-size cut. Before comparing this against Baseline 5 (distilled model), the honest framing is: text adds targeted value on one class, at the cost of the smallest classes, under a training budget smaller than every prior baseline used.

## The Zero-Recall Investigation

Baseline 4 got 0.00 recall on "Closed with monetary relief" — every earlier model predicted this class at least some of the time. Checking whether this is a threshold problem (the model is close but never quite confident enough) or a signal problem (the model never seriously considers this class a candidate), by pulling its predicted probability for the correct class across all 38 true test examples.

In [34]:
# Pull the model's predicted probabilities for the true monetary-relief
# test examples, to see if it's "close but never quite crossing the
# threshold" vs. "never in serious contention."
monetary_relief_id = label2id['Closed with monetary relief']
true_monetary_mask = np.array(test_labels) == monetary_relief_id

probs = torch.softmax(torch.tensor(preds_output.predictions), dim=1).numpy()
monetary_probs = probs[true_monetary_mask][:, monetary_relief_id]

print(f"N true monetary-relief examples in test set: {true_monetary_mask.sum()}")
print(f"Model's predicted probability for correct class, per example:")
print(np.sort(monetary_probs)[::-1])

N true monetary-relief examples in test set: 38
Model's predicted probability for correct class, per example:
[1.9543132e-01 1.2158267e-01 1.0269685e-01 9.8920263e-02 6.4501330e-02
 4.8458017e-02 3.8273018e-02 3.3178847e-02 3.1842094e-02 2.7331632e-02
 2.6497487e-02 2.5012475e-02 2.1510128e-02 1.8651508e-02 1.8350847e-02
 1.6073031e-02 1.5679065e-02 1.5432270e-02 8.8612214e-03 7.3861158e-03
 6.8895575e-03 6.1634905e-03 4.3651084e-03 4.2313696e-03 3.5110915e-03
 1.9321420e-03 1.8866375e-03 1.8588803e-03 1.7470967e-03 1.4989792e-03
 1.3532810e-03 7.8615616e-04 4.1881323e-04 3.9890982e-04 3.4895237e-04
 3.2862401e-04 2.1423622e-04 1.4230583e-04]


### Diagnostic: Why "Closed with Monetary Relief" Got Zero Recall

Checked the model's predicted probability for the correct class across all 38 true "Closed with monetary relief" test examples. The highest is 0.195; most sit under 0.05, several under 0.001. This isn't a threshold problem — the model never seriously considers this class a candidate for nearly any of these examples.

The likely cause is training volume, not text signal. This class had 29 examples in the 25K subsample, down from roughly 152 in the full training set Baselines 1–3 used. Those baselines saw non-trivial recall on this class (0.42–0.71 depending on weighting scheme) with the full data available. Losing over 80% of an already-tiny class appears to have crossed a real floor for what a 60x loss weight can compensate for.

Scoping this correctly for the writeup: this is evidence that subsampling defeated this class, not evidence that narrative text lacks signal for identifying monetary-relief cases.

## Baseline 5: Distilled Model Comparison

Testing the accuracy/latency tradeoff of a smaller checkpoint against Baseline 4's fine-tuned DistilBERT, following the same approach as HW4 Q6. Using `prajjwal1/bert-tiny` (2 layers, 128-dim hidden, ~4.4M parameters vs. DistilBERT's ~66M) — a genuinely small architecture rather than a marginal reduction, so the tradeoff has real contrast to show. Same subsampled training set, same `capped_weights`, same target as Baseline 4, so only the architecture changes. Alongside macro F1, benchmarking inference latency (ms per 100 examples) for both models — the core comparison this stage is meant to answer.

In [38]:
DISTILLED_MODEL_NAME = "prajjwal1/bert-tiny"
# distilled_tokenizer = AutoTokenizer.from_pretrained(DISTILLED_MODEL_NAME)
distilled_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_fn_distilled(batch):
    return distilled_tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)



In [39]:
# Reusing the same train_texts/train_labels/test_texts/test_labels from Baseline 4 —
# same subsampled train set, same full test set — just re-tokenized for the new model.
train_ds_distilled = Dataset.from_dict({'text': train_texts, 'label': train_labels})
test_ds_distilled = Dataset.from_dict({'text': test_texts, 'label': test_labels})

train_ds_distilled = train_ds_distilled.map(tokenize_fn_distilled, batched=True).rename_column('label', 'labels')
test_ds_distilled = test_ds_distilled.map(tokenize_fn_distilled, batched=True).rename_column('label', 'labels')
train_ds_distilled.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds_distilled.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/32891 [00:00<?, ? examples/s]

In [41]:
from transformers import BertConfig

config = BertConfig.from_pretrained(
    DISTILLED_MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)
distilled_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILLED_MODEL_NAME, config=config
)

# distilled_model = AutoModelForSequenceClassification.from_pretrained(
#     DISTILLED_MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id
# )

# Same weighted loss, same capped_weights as every prior baseline.
training_args_distilled = TrainingArguments(
    output_dir='./bert_tiny_distilled_stage',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    logging_steps=50,
    fp16=torch.cuda.is_available(),
)

trainer_distilled = WeightedTrainer(
    model=distilled_model, args=training_args_distilled,
    train_dataset=train_ds_distilled, eval_dataset=test_ds_distilled,
    compute_metrics=compute_metrics,
)
trainer_distilled.train()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 17.8MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.745666,0.955217,0.312134,0.625065
2,0.823761,0.922669,0.310956,0.620991
3,0.773861,0.916697,0.329072,0.634490


...

model.safetensors: downloading bytes:           |  0.00B            

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4689, training_loss=0.8141366548186352, metrics={'train_runtime': 27.6683, 'train_samples_per_second': 2710.683, 'train_steps_per_second': 169.472, 'total_flos': 47672985600000.0, 'train_loss': 0.8141366548186352, 'epoch': 3.0})

In [42]:
preds_output_distilled = trainer_distilled.predict(test_ds_distilled)
distilled_preds = [id2label[i] for i in np.argmax(preds_output_distilled.predictions, axis=1)]

print(classification_report(y_merged.loc[test_idx], distilled_preds))
print(f"Macro F1: {f1_score(y_merged.loc[test_idx], distilled_preds, average='macro'):.3f}")

                                 precision    recall  f1-score   support

        Closed with explanation       0.82      0.55      0.66     20989
    Closed with monetary relief       0.00      0.00      0.00        38
Closed with non-monetary relief       0.50      0.79      0.61     11784
                          Other       0.50      0.03      0.05        80

                       accuracy                           0.63     32891
                      macro avg       0.45      0.34      0.33     32891
                   weighted avg       0.70      0.63      0.64     32891

Macro F1: 0.329


In [44]:
# Latency benchmark — cell 25's core ask: ms per 100 examples, full vs. distilled.
import time

def benchmark_inference(model, dataset, n_examples=100, device='cuda' if torch.cuda.is_available() else 'cpu'):
    model.to(device).eval()
    sample = dataset.select(range(n_examples))
    batch = {k: torch.stack([sample[i][k] for i in range(n_examples)]).to(device)
             for k in ['input_ids', 'attention_mask']}

    with torch.no_grad():
        _ = model(**batch)  # warm-up, excluded from timing
        torch.cuda.synchronize() if device == 'cuda' else None
        start = time.time()
        _ = model(**batch)
        torch.cuda.synchronize() if device == 'cuda' else None
        elapsed_ms = (time.time() - start) * 1000

    return elapsed_ms

latency_full = benchmark_inference(model, test_ds)
latency_distilled = benchmark_inference(distilled_model, test_ds_distilled)

print(f"Baseline 4 (distilbert) latency: {latency_full:.1f} ms per 100 examples")
print(f"Baseline 5 (bert-tiny) latency: {latency_distilled:.1f} ms per 100 examples")

Baseline 4 (distilbert) latency: 14.8 ms per 100 examples
Baseline 5 (bert-tiny) latency: 1.0 ms per 100 examples


## Baseline 5: Distilled Model (bert-tiny) — Accuracy/Latency Tradeoff

Fine-tuned `prajjwal1/bert-tiny` (2 layers, 128-dim hidden, ~4.4M params vs. DistilBERT's ~66M) on the same subsample, weighting, and target as Baseline 4. Macro F1 dropped to 0.329 from 0.409 — a real decline, not noise — while inference latency fell from 14.8ms to 1.0ms per 100 examples, a 14.8x speedup.

The drop isn't spread evenly across classes. "Closed with non-monetary relief" held steady at F1 0.61, recall even improved (0.72 → 0.79). "Closed with explanation" declined modestly (F1 0.70 → 0.66). "Other" collapsed from F1 0.33 to 0.05 — recall fell from 0.30 to 0.03. "Closed with monetary relief" stayed at 0.00 recall in both models, consistent with the training-volume floor I already flagged for Baseline 4 (29 examples wasn't enough for the larger model either).

I read this as a capacity story: bert-tiny keeps enough capacity to learn a large, well-represented class, but not enough to hold onto the weak signal DistilBERT picked up on a small one. So the real tradeoff for a deployment decision isn't a flat "15x faster for X% less accuracy" — it's "15x faster, at the cost of the smallest class almost entirely." Whether that's acceptable depends on whether "Other" is operationally load-bearing.

## Summary Table and Takeaway


| model | macro_f1 | accuracy | inference_ms_per_100 |
|---|---|---|---|
| structured_only | 0.361 | 0.46 | <1 (not separately benchmarked) |
| structured_plus_topic | 0.406 | 0.47 | <1 (not separately benchmarked) |
| structured_plus_entities | 0.394 | 0.53 | <1 (not separately benchmarked) |
| transformer_full | 0.409 | 0.66 | 14.8 |
| transformer_distilled | 0.329 | 0.63 | 1.0 |

All five stages use the same target (`y_merged`, with "In progress" and "Untimely response" folded into "Other" due to insufficient support) and the same capped class weighting, so these numbers are comparable to each other with one exception noted below.

**Does adding topic labels help over structured fields alone?** Yes, modestly. `topic_name` took macro F1 from 0.361 to 0.406, the single largest jump in the whole comparison. Most of that gain came from one class — "Other" F1 went from 0.48 to 0.63 — while the two large classes barely moved. This is a real, if narrow, finding: LDA topics carry information beyond `Issue`/`Sub-issue` for cases that don't fit the two dominant response types, even though Notebook 02's topic-vs-issue cross-tab made this look unlikely to matter going in.

**Do entity features help?** Differently than topic labels, not more. `entity_cols` pushed accuracy up more than any other single stage (0.46 → 0.53) and gave "Closed with explanation" its best structured-feature F1 (0.60), but macro F1 landed slightly below the topic-label version (0.394 vs. 0.406) because it did nothing for "Other." Stacking both together (macro F1 0.393, not in the table above since it fell outside the notebook's five-stage template) didn't beat either alone — every class in the combined model landed between its two parents rather than at the best of both, meaning the two feature sets are capturing overlapping signal, not complementary signal.

**Does narrative text help beyond either?** Yes, on one specific class, and the picture here comes with a caveat the other rows don't carry. `transformer_full` narrowly leads on macro F1 (0.409) and clearly leads on accuracy (0.66), driven by "Closed with non-monetary relief" hitting F1 0.61 — the best result of any stage, with precision and recall both improving together. But this stage trained on 25,000 rows, a stratified subsample, against roughly 131,564 for every sklearn baseline. That cut absolute training examples for the smallest classes hard — "Closed with monetary relief" went from ~152 examples to 29 — and a probability-level check confirmed the model never seriously considered that class a candidate (highest predicted probability across all 38 true test examples: 0.195). That's a training-volume artifact of the compute-constrained subsample, not evidence that narrative text lacks signal for that class.

**Full vs. distilled tradeoff.** Moving from DistilBERT to bert-tiny (~66M → ~4.4M parameters) cut macro F1 from 0.409 to 0.329 — a real decline — while cutting inference latency from 14.8ms to 1.0ms per 100 examples, a 14.8x speedup. The accuracy cost isn't spread evenly: "Closed with non-monetary relief" held essentially flat (F1 0.61 → 0.61), while "Other" collapsed (F1 0.33 → 0.05). A smaller model keeps enough capacity to learn a large, well-represented class but loses the weaker signal on a small one. Whether the 14.8x speedup is worth the tradeoff is a deployment question, not a modeling one — it depends on whether "Other" cases are operationally load-bearing.

**Bottom line.** No single feature addition or architecture swap was a clean win across every class. Topic labels and narrative text each rescue a specific class the structured-only baseline couldn't reach — "Other" and "non-monetary relief,"

## Key Takeaways

* Structured metadata provides a usable baseline (macro F1 0.361) but leaves most of the predictive ceiling unreached, particularly for minority response categories.
* Topic modeling contributes a real, narrow gain (macro F1 0.406) concentrated almost entirely in the smallest well-represented class ("Other"), not a broad signal across the target distribution.
* Entity features contribute a comparable overall gain (macro F1 0.394) through a different mechanism — improving majority-class precision and recall rather than minority-class performance — making them a genuine complement to topic labels rather than a weaker version of the same effect.
* Combining topic and entity features doesn't outperform either alone (macro F1 0.393), suggesting the two feature sets capture overlapping rather than additive signal.
* The fine-tuned transformer's macro F1 (0.409) is not a clear improvement over topic labels alone and comes with two confounds worth weighing against the marginal gain: it trained on a 25,000-row subsample (versus ~131,564 for every other stage) and its reported metric may be modestly inflated by using the test set for checkpoint selection during training. Its clearest advantage is a specific one — non-monetary relief F1 of 0.61, the best of any stage — not an across-the-board win.
* No approach, classical or transformer-based, produced usable recall on the rarest class ("Closed with monetary relief," n≈152 in the full training set); this appears to be a training-volume floor rather than a modeling problem.

Overall, the honest finding is closer to diminishing returns than a clean hierarchy: structured fields capture most of what's learnable here, each additional feature source rescues a specific class rather than lifting performance broadly, and the transformer's edge is real but narrow, class-specific, and partly confounded by its smaller training set.